In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

raiz = Path.cwd().resolve()
if not (raiz / "dados").is_dir():
    raiz = raiz.parent
if not (raiz / "dados").is_dir():
    raise FileNotFoundError("Execute o notebook a partir da raiz do projeto ou da pasta script.")
sys.path.insert(0, str(raiz / "src"))

from preprocessamento import (carregar_dados, preparar_dados, criar_preprocessador,
                              criar_particoes, NUMERICAS, CATEGORICAS)
from avaliacao import avaliar_modelo

saida = raiz / "resultados"
saida.mkdir(exist_ok=True)
caminho = raiz / "dados" / "online_shoppers_convertido.csv"
dados = carregar_dados(caminho)
print("Dimensões da base convertida:", dados.shape)
print("Duplicatas exatas:", dados.duplicated().sum())
print("Valores ausentes:", dados.isna().sum().sum())
display(dados.head())


In [ ]:
X, y = preparar_dados(dados)
particoes = criar_particoes(X, y)
print("Registros após remover duplicatas:", len(X))
print("Atributos numéricos:", NUMERICAS)
print("Atributos categóricos:", CATEGORICAS)
print("Distribuição após remoção das duplicatas:")
display(y.value_counts().sort_index().rename(index={0: "Sem compra", 1: "Compra"}).to_frame("quantidade"))
print("Número de partições estratificadas:", len(particoes))


## KNN: duas configurações

São comparados `k=5` com pesos uniformes e `k=11` com pesos pela distância. Ambos usam as mesmas cinco partições estratificadas, com embaralhamento e semente 42, que também devem ser usadas pelos demais classificadores do grupo. Não há reamostragem. Métricas: acurácia, precisão, revocação e F1 para a classe compra. As tabelas mostram os valores por fold e a média e o desvio padrão entre folds.


In [ ]:
configuracoes = {
    "KNN k=5, uniforme": KNeighborsClassifier(n_neighbors=5, weights="uniform"),
    "KNN k=11, distância": KNeighborsClassifier(n_neighbors=11, weights="distance"),
}
resultados = []
matrizes = {}

for nome, classificador in configuracoes.items():
    # O preprocessador fica dentro do pipeline para ser ajustado em cada treino.
    modelo = Pipeline([
        ("preprocessamento", criar_preprocessador()),
        ("modelo", classificador),
    ])
    metricas, matrizes[nome] = avaliar_modelo(modelo, X, y, particoes)
    resultados.append(metricas.reset_index().assign(configuracao=nome))

# Mantém os nomes e a estrutura dos arquivos produzidos anteriormente.
por_fold = pd.concat(resultados, ignore_index=True).rename(columns={
    "Fold": "fold", "Acurácia": "acuracia", "Precisão": "precisao",
    "Recall": "revocacao", "F1-score": "f1",
})
por_fold = por_fold[["configuracao", "fold", "acuracia", "precisao", "revocacao", "f1"]]
resumo = por_fold.groupby("configuracao")[["acuracia", "precisao", "revocacao", "f1"]].agg(["mean", "std"])
print("Métricas por partição:")
display(por_fold.round(4))
print("Média e desvio padrão entre as cinco partições:")
display(resumo.round(4))
por_fold.to_csv(saida / "knn_metricas_por_fold.csv", index=False, encoding="utf-8-sig")
resumo.to_csv(saida / "knn_metricas_resumo.csv", encoding="utf-8-sig")


In [ ]:
fig, eixos = plt.subplots(1, len(matrizes), figsize=(11, 4))
for eixo, (nome, matriz), arquivo in zip(
    eixos, matrizes.items(), ["matriz_knn_5.csv", "matriz_knn_11.csv"]
):
    ConfusionMatrixDisplay(matriz, display_labels=["Sem compra", "Compra"]).plot(
        ax=eixo, cmap="Blues", colorbar=False, values_format="d"
    )
    eixo.set_title(nome)
    eixo.set_xlabel("Previsto")
    eixo.set_ylabel("Real")
    pd.DataFrame(matriz, index=["real_sem_compra", "real_compra"],
                 columns=["previsto_sem_compra", "previsto_compra"]).to_csv(
                     saida / arquivo, encoding="utf-8-sig")
plt.tight_layout()
plt.savefig(saida / "matrizes_confusao_knn.png", dpi=300)
plt.show()
